In [2]:
import pandas as pd
import networkx as nx
from google.colab import files

print("1. Cargando archivos originales...")
df_nodos = pd.read_csv("nodos_gephi_final.csv")
df_relaciones = pd.read_csv("relaciones_gephi_final.csv")

print(f"Nodos iniciales: {len(df_nodos)} | Relaciones iniciales: {len(df_relaciones)}")

# ==========================================
# PASO 1: Calcular la cantidad de aristas (Grado) por nodo
# ==========================================
# Contamos cuántas veces aparece cada ID como Source o Target
conteo_source = df_relaciones['Source'].value_counts()
conteo_target = df_relaciones['Target'].value_counts()
grados = conteo_source.add(conteo_target, fill_value=0)

# Mapeamos el grado al dataframe de nodos
df_nodos['grado'] = df_nodos['Id'].map(grados).fillna(0)

# ==========================================
# PASO 2: Eliminar "Callejones sin salida" vacíos
# ==========================================
# Criterio del usuario: Eliminar nodos que NO tienen datos (época ni ubicación)
# y que ADEMÁS tienen grado 1 (no sirven de puente estructural).
sin_datos = (df_nodos['epoca'] == 'Desconocido') & (df_nodos['ubicacion'] == 'Desconocido')
grado_irrelevante = df_nodos['grado'] <= 1

mascara_basura = sin_datos & grado_irrelevante
nodos_limpios = df_nodos[~mascara_basura].copy()

print(f"-> Se eliminaron {mascara_basura.sum()} nodos 'basura' (sin datos y con 1 sola arista).")

# ==========================================
# PASO 3: Filtrar Micro-Islas (Análisis de Componentes con NetworkX)
# ==========================================
# Al eliminar nodos basura, algunas aristas quedan huérfanas. Actualizamos relaciones:
ids_validos = set(nodos_limpios['Id'])
relaciones_limpias = df_relaciones[df_relaciones['Source'].isin(ids_validos) &
                                   df_relaciones['Target'].isin(ids_validos)].copy()

print("2. Analizando la topología de la red...")
G = nx.Graph()
# Creamos el grafo solo con las relaciones válidas
G.add_edges_from(zip(relaciones_limpias['Source'], relaciones_limpias['Target']))

# Identificamos las "islas" (componentes conectados)
componentes = sorted(nx.connected_components(G), key=len, reverse=True)

# Nos quedamos SOLO con las macro-redes (islas que tengan más de 100 personas).
# Esto mantendrá la dinastía principal europea y quizás la asiática/islámica grande,
# borrando las familias de 2 o 3 personas.
nodos_estructurales = set()
for comp in componentes:
    if len(comp) > 100:
        nodos_estructurales.update(comp)

print(f"-> Se encontraron {len(componentes)} islas en total.")
print(f"-> Se conservarán los nodos de las macro-redes principales.")

# Aplicar el filtro final de topología
df_nodos_final = nodos_limpios[nodos_limpios['Id'].isin(nodos_estructurales)].drop(columns=['grado'])
df_relaciones_final = relaciones_limpias[relaciones_limpias['Source'].isin(nodos_estructurales) &
                                         relaciones_limpias['Target'].isin(nodos_estructurales)]

print(f"\n3. REDUCCIÓN FINAL COMPLETA:")
print(f"Nodos finales: {len(df_nodos_final)} (Reducción del {100 - (len(df_nodos_final)/len(df_nodos)*100):.1f}%)")
print(f"Relaciones finales: {len(df_relaciones_final)} (Reducción del {100 - (len(df_relaciones_final)/len(df_relaciones)*100):.1f}%)")

# ==========================================
# PASO 4: Exportar y Descargar
# ==========================================
df_nodos_final.to_csv("nodos_gephi_optimizados.csv", index=False, encoding="utf-8")
df_relaciones_final.to_csv("relaciones_gephi_optimizadas.csv", index=False, encoding="utf-8")

files.download("nodos_gephi_optimizados.csv")
files.download("relaciones_gephi_optimizadas.csv")

1. Cargando archivos originales...
Nodos iniciales: 33915 | Relaciones iniciales: 99584
-> Se eliminaron 4710 nodos 'basura' (sin datos y con 1 sola arista).
2. Analizando la topología de la red...
-> Se encontraron 1870 islas en total.
-> Se conservarán los nodos de las macro-redes principales.

3. REDUCCIÓN FINAL COMPLETA:
Nodos finales: 18807 (Reducción del 44.5%)
Relaciones finales: 82463 (Reducción del 17.2%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>